# 09 - Model Comparison: RoBERTa Fine-Tuned vs Ollama Zero-Shot

This notebook loads all saved results from notebooks 07 and 08 and produces a comprehensive comparison.

### What we compare
| Model | Type | Training data needed |
|---|---|---|
| RoBERTa-base fine-tuned | Fine-tuned | Yes (labeled gold standard) |
| Ollama llama3.1:8b zero-shot | Zero-shot | No |

### Datasets: Manchester, Monkeypox, PHEME

### Metrics: F1 Macro, F1 Weighted, Accuracy, Precision, Recall, F1 (positive class)

## 1. Imports & Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from sklearn.metrics import confusion_matrix

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size']  = 11

RESULTS_DIR = Path('../../results')
PREDS_DIR   = RESULTS_DIR / 'predictions'
FIGS_DIR    = RESULTS_DIR / 'figures'
FIGS_DIR.mkdir(parents=True, exist_ok=True)

DATASETS = ['manchester', 'monkeypox', 'pheme']
LABEL_CONFIGS = {
    'manchester': {'label_names': ['reliable', 'misinformation'], 'pos_label': 'misinformation', 'label_map': {'reliable': 0, 'misinformation': 1}},
    'monkeypox':  {'label_names': ['reliable', 'misinformation'], 'pos_label': 'misinformation', 'label_map': {'reliable': 0, 'misinformation': 1}},
    'pheme':      {'label_names': ['not_rumour', 'rumour'],       'pos_label': 'rumour',          'label_map': {'not_rumour': 0, 'rumour': 1}},
}

COLORS = {'roberta': '#2980b9', 'zeroshot': '#e67e22'}

print("Setup complete.")

## 2. Load All Results

In [ ]:
def load_summaries():
    """Load summary CSVs saved by notebooks 07 and 08."""
    rows = []
    for ds in DATASETS:
        roberta_path  = PREDS_DIR / f'{ds}_roberta_summary.csv'
        zeroshot_path = PREDS_DIR / f'{ds}_zeroshot_summary.csv'

        if roberta_path.exists():
            r = pd.read_csv(roberta_path).iloc[0].to_dict()
            r['model_type'] = 'RoBERTa Fine-Tuned'
            r['dataset'] = ds
            rows.append(r)
        else:
            print(f"WARNING: {roberta_path} not found — run notebook 07 first")

        if zeroshot_path.exists():
            z = pd.read_csv(zeroshot_path).iloc[0].to_dict()
            z['model_type'] = 'Ollama Zero-Shot'
            z['dataset'] = ds
            rows.append(z)
        else:
            print(f"WARNING: {zeroshot_path} not found — run notebook 08 first")

    return pd.DataFrame(rows)


df_summary = load_summaries()
print(f"Loaded {len(df_summary)} result rows")
print(df_summary[['dataset', 'model_type', 'test_f1_macro', 'test_accuracy']].to_string(index=False))

## 3. Main Comparison Table

In [ ]:
METRIC_COLS = ['test_f1_macro', 'test_f1_weighted', 'test_accuracy', 'test_precision', 'test_recall']

# Pivot for easy comparison
pivot = df_summary.pivot_table(
    index='dataset',
    columns='model_type',
    values=METRIC_COLS
).round(4)

print("\n=== FULL COMPARISON TABLE ===")
print(pivot.to_string())

# Delta: RoBERTa - ZeroShot (positive = RoBERTa wins)
print("\n=== DELTA (RoBERTa Fine-Tuned MINUS Ollama Zero-Shot) ===")
for col in METRIC_COLS:
    try:
        delta = pivot[col]['RoBERTa Fine-Tuned'] - pivot[col]['Ollama Zero-Shot']
        print(f"\n{col}:")
        print(delta.round(4).to_string())
    except KeyError:
        pass

## 4. Grouped Bar Chart — F1 Macro Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=False)

metrics_to_plot = ['test_f1_macro', 'test_accuracy', 'test_precision', 'test_recall']
metric_labels   = ['F1 Macro', 'Accuracy', 'Precision (Macro)', 'Recall (Macro)']

for ax, ds in zip(axes, DATASETS):
    ds_data = df_summary[df_summary['dataset'] == ds]

    roberta_row  = ds_data[ds_data['model_type'] == 'RoBERTa Fine-Tuned']
    zeroshot_row = ds_data[ds_data['model_type'] == 'Ollama Zero-Shot']

    x = np.arange(len(metrics_to_plot))
    width = 0.35

    r_vals = [roberta_row[m].values[0]  if len(roberta_row)  > 0 and m in roberta_row.columns  else 0 for m in metrics_to_plot]
    z_vals = [zeroshot_row[m].values[0] if len(zeroshot_row) > 0 and m in zeroshot_row.columns else 0 for m in metrics_to_plot]

    bars1 = ax.bar(x - width/2, r_vals, width, label='RoBERTa Fine-Tuned', color=COLORS['roberta'],  alpha=0.85, edgecolor='black', linewidth=0.5)
    bars2 = ax.bar(x + width/2, z_vals, width, label='Ollama Zero-Shot',   color=COLORS['zeroshot'], alpha=0.85, edgecolor='black', linewidth=0.5)

    for bar in bars1 + bars2:
        h = bar.get_height()
        if h > 0:
            ax.text(bar.get_x() + bar.get_width()/2, h + 0.005, f'{h:.3f}',
                    ha='center', va='bottom', fontsize=7.5, rotation=45)

    ax.set_title(ds.upper(), fontweight='bold', fontsize=13)
    ax.set_xticks(x)
    ax.set_xticklabels(metric_labels, rotation=20, ha='right', fontsize=9)
    ax.set_ylim([0, 1.1])
    ax.set_ylabel('Score')
    ax.legend(fontsize=8)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('RoBERTa Fine-Tuned vs Ollama Zero-Shot (llama3.1) - All Datasets', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
fig_path = FIGS_DIR / 'comparison_grouped_bars.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {fig_path}")

## 5. Heatmap — F1 Macro Across All Conditions

In [ ]:
# Build heatmap matrix
heatmap_cols = ['test_f1_macro', 'test_f1_weighted', 'test_accuracy', 'test_precision', 'test_recall']
heatmap_labels = ['F1 Macro', 'F1 Weighted', 'Accuracy', 'Precision', 'Recall']

rows_r, rows_z = [], []
idx_r, idx_z = [], []

for ds in DATASETS:
    r = df_summary[(df_summary['dataset'] == ds) & (df_summary['model_type'] == 'RoBERTa Fine-Tuned')]
    z = df_summary[(df_summary['dataset'] == ds) & (df_summary['model_type'] == 'Ollama Zero-Shot')]
    if len(r):
        rows_r.append([r[c].values[0] if c in r.columns else np.nan for c in heatmap_cols])
        idx_r.append(f'RoBERTa | {ds}')
    if len(z):
        rows_z.append([z[c].values[0] if c in z.columns else np.nan for c in heatmap_cols])
        idx_z.append(f'Ollama | {ds}')

all_rows = rows_r + rows_z
all_idx  = idx_r  + idx_z

hm_df = pd.DataFrame(all_rows, index=all_idx, columns=heatmap_labels)

fig, ax = plt.subplots(figsize=(10, max(4, len(all_idx) * 0.7)))
sns.heatmap(
    hm_df.round(3),
    annot=True, fmt='.3f', cmap='YlGnBu',
    vmin=0.5, vmax=1.0,
    linewidths=0.5, linecolor='white',
    ax=ax, annot_kws={'size': 10}
)
ax.set_title('Model Comparison Heatmap — All Metrics & Datasets', fontsize=13, fontweight='bold')
ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha='right')
plt.tight_layout()
fig_path = FIGS_DIR / 'comparison_heatmap.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {fig_path}")

## 6. Confusion Matrix Side-by-Side

In [ ]:
fig, axes = plt.subplots(len(DATASETS), 2, figsize=(12, 5 * len(DATASETS)))

for row_idx, ds in enumerate(DATASETS):
    cfg = LABEL_CONFIGS[ds]

    roberta_pred_path  = PREDS_DIR / f'{ds}_roberta_test_predictions.csv'
    zeroshot_pred_path = PREDS_DIR / f'{ds}_zeroshot_test_predictions.csv'

    for col_idx, (pred_path, title, cmap) in enumerate([
        (roberta_pred_path,  'RoBERTa Fine-Tuned',    'Blues'),
        (zeroshot_pred_path, 'Ollama Zero-Shot',       'Oranges'),
    ]):
        ax = axes[row_idx][col_idx]
        if pred_path.exists():
            df_p = pd.read_csv(pred_path)
            y_true = df_p['true_label_int'].values
            y_pred = df_p['pred_label_int'].values
            cm = confusion_matrix(y_true, y_pred)
            cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

            annot = np.array([
                [f"{cm[i,j]}\n({cm_norm[i,j]:.1%})" for j in range(cm.shape[1])]
                for i in range(cm.shape[0])
            ])
            sns.heatmap(
                cm_norm, annot=annot, fmt='', cmap=cmap,
                xticklabels=cfg['label_names'],
                yticklabels=cfg['label_names'],
                ax=ax, vmin=0, vmax=1
            )
            ax.set_title(f'{ds.upper()} | {title}', fontweight='bold')
            ax.set_xlabel('Predicted')
            ax.set_ylabel('True')
        else:
            ax.text(0.5, 0.5, f'No data\n{pred_path.name}',
                    ha='center', va='center', transform=ax.transAxes)
            ax.set_title(f'{ds.upper()} | {title}', fontweight='bold')

plt.suptitle('Confusion Matrices: RoBERTa Fine-Tuned vs Ollama Zero-Shot', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
fig_path = FIGS_DIR / 'comparison_confusion_matrices.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {fig_path}")

## 7. Per-Dataset Winner Analysis

In [ ]:
print("=" * 70)
print(" WINNER ANALYSIS — F1 Macro (primary metric)")
print("=" * 70)

for ds in DATASETS:
    r = df_summary[(df_summary['dataset'] == ds) & (df_summary['model_type'] == 'RoBERTa Fine-Tuned')]
    z = df_summary[(df_summary['dataset'] == ds) & (df_summary['model_type'] == 'Ollama Zero-Shot')]

    if len(r) and len(z) and 'test_f1_macro' in r.columns and 'test_f1_macro' in z.columns:
        r_f1 = r['test_f1_macro'].values[0]
        z_f1 = z['test_f1_macro'].values[0]
        delta = r_f1 - z_f1
        winner = 'RoBERTa Fine-Tuned' if delta > 0 else 'Ollama Zero-Shot'
        sign = '+' if delta > 0 else ''
        print(f"\n  {ds.upper():12s} → Winner: {winner}")
        print(f"               RoBERTa F1:  {r_f1:.4f}")
        print(f"               Ollama F1:   {z_f1:.4f}")
        print(f"               Delta:       {sign}{delta:.4f}")
    else:
        print(f"\n  {ds.upper():12s} → Missing data (run notebooks 07 and 08 first)")

## 8. Overall Summary Table

In [ ]:
fig, ax = plt.subplots(figsize=(14, max(4, (len(df_summary) + 1) * 0.55)))
ax.axis('off')

col_labels = ['Dataset', 'Model', 'F1 Macro', 'F1 Weighted', 'Accuracy', 'Precision', 'Recall']

table_data = []
for ds in DATASETS:
    for mtype in ['RoBERTa Fine-Tuned', 'Ollama Zero-Shot']:
        row = df_summary[(df_summary['dataset'] == ds) & (df_summary['model_type'] == mtype)]
        if len(row):
            r = row.iloc[0]
            table_data.append([
                ds.capitalize(),
                mtype,
                f"{r.get('test_f1_macro', np.nan):.4f}",
                f"{r.get('test_f1_weighted', np.nan):.4f}",
                f"{r.get('test_accuracy', np.nan):.4f}",
                f"{r.get('test_precision', np.nan):.4f}",
                f"{r.get('test_recall', np.nan):.4f}",
            ])

if table_data:
    table = ax.table(
        cellText=table_data,
        colLabels=col_labels,
        cellLoc='center',
        loc='center',
        bbox=[0, 0, 1, 1]
    )
    table.auto_set_font_size(False)
    table.set_fontsize(10)

    for (row, col), cell in table.get_celld().items():
        if row == 0:
            cell.set_facecolor('#2c3e50')
            cell.set_text_props(color='white', fontweight='bold')
        elif row > 0:
            model_val = table_data[row - 1][1] if row <= len(table_data) else ''
            if 'RoBERTa' in model_val:
                cell.set_facecolor('#d6eaf8')
            else:
                cell.set_facecolor('#fdebd0')

    table.auto_set_column_width(col=list(range(len(col_labels))))
else:
    ax.text(0.5, 0.5, 'No results loaded. Run notebooks 07 and 08 first.',
            ha='center', va='center', fontsize=12, transform=ax.transAxes)

ax.set_title('Complete Results: RoBERTa Fine-Tuned vs Ollama Zero-Shot (llama3.1)', fontsize=13, fontweight='bold', pad=20)
plt.tight_layout()
fig_path = FIGS_DIR / 'comparison_final_table.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {fig_path}")

## 9. Save Master Results CSV

In [ ]:
master_path = RESULTS_DIR / 'master_results.csv'
df_summary.to_csv(master_path, index=False)
print(f"Master results saved: {master_path}")
print("\nAll comparisons complete!")
print("\nFigures saved:")
for f in sorted(FIGS_DIR.glob('comparison_*.png')):
    print(f"  {f.name}")